In [1]:
import nibabel as nib
import numpy as np

In [2]:
input_file = '/Users/fei/Desktop/Harris_Lab/dFNC/ICA_Results/All_infomax/07d/55c/all_07d_55c_mean_component_ica_s_all_.nii'
img = nib.load(input_file)
data = img.get_fdata()
header = img.header
affine = img.affine

In [5]:
# # Z-Score Converter, use prior to ROI identification if needed
# flattened_data = data.flatten()

# mean = np.mean(flattened_data)
# std = np.std(flattened_data)

# z_scores = (data - mean) / std

# z_img = nib.Nifti1Image(z_scores, affine, header)

# output_file = 'all_07d_15c_mean_component_ica_s_all_z_scores.nii'
# nib.save(z_img, output_file)

In [32]:
data_3d = np.split(data, data.shape[3], axis=3)
data_3d = [np.squeeze(i) for i in data_3d]
print(f"Number of components: {len(data_3d)}")
print(f"Component dimensions: {data_3d[0].shape}")

Number of components: 55
Component dimensions: (96, 96, 25)


In [125]:
atlas_file = '01_study_specific_atlas_relabel.nii'
atlas = nib.load(atlas_file)
atlas_data = atlas.get_fdata()
print(atlas_data.shape)

(96, 96, 25)


In [127]:
assignments = {}

for d in range(len(data_3d)):
    assignments[d + 1] = set()
    rois = {}
    for i in range(96):
        for j in range(96):
            for k in range(25):
                if data_3d[d][i, j, k] >= 2.33: # increase to 3.01 if needed
                        rois[data_3d[d][i, j, k]] = [i, j, k]
    rois_list = list(rois.keys())
    rois_list.sort(reverse=True)
    sorted_rois = {r: rois[r] for r in rois_list}
    for key, value in sorted_rois.items():
        if value == 0:
            del sorted_rois[key]
    for i, (key, value) in enumerate(sorted_rois.items()):
        if atlas_data[value[0], value[1], value[2]] != 0:
            assignments[d + 1].add(int(atlas_data[value[0], value[1], value[2]]))
        if len(assignments[d + 1]) == 10:
            assignments[d + 1] = sorted(assignments[d + 1])
            break

for a in assignments:
    print(f"{a}: {assignments[a]}")

1: [10, 11, 12, 19, 46, 71, 85, 106, 117, 124]
2: [2, 16, 20, 22, 39, 51, 52, 53, 55, 57]
3: [5, 18, 33, 36, 46, 60, 61, 65, 66, 99]
4: [1, 4, 10, 12, 36, 39, 42, 78, 80, 99]
5: [13, 21, 34, 38, 40, 44, 81, 100, 103, 107]
6: [2, 14, 16, 22, 57, 58, 73, 75, 82, 88]
7: [2, 3, 4, 6, 14, 22, 42, 61, 119, 120]
8: [1, 6, 42, 59, 71, 72, 77, 103, 118, 124]
9: [7, 10, 25, 27, 29, 32, 46, 62, 78, 108]
10: [37, 39, 47, 52, 53, 54, 56, 104, 114, 115]
11: [5, 6, 33, 59, 60, 65, 66, 71, 86, 99]
12: [23, 31, 32, 37, 45, 64, 67, 68, 69, 70]
13: [10, 12, 23, 27, 29, 32, 35, 46, 78, 79]
14: [12, 21, 35, 48, 63, 79, 80, 101, 108, 109]
15: [11, 21, 26, 28, 31, 35, 46, 48, 63, 101]


In [129]:
roi_names = []
with open('rois_labels_not_numbered.txt', 'r') as rois_labels:
    for label in rois_labels:
        roi_names.append(label)

In [161]:
comp = 15
print(f"Component {comp} ROIs:\n")
for roi in assignments[comp]:
    print(roi_names[roi - 1])

Component 15 ROIs:

Central_Gray_left 

Genu_corpus_callosum_left

Hipp_CA3_A_left

Hipp_dentegyrus_A_left

Hipp_Sub_A_left

Inferior_Colliculus_left

Rest_of_Midbrain_left

Retrosplenial granular b cortex (left)

Superior_Colliculus_left

Inferior_Colliculus_right

